# Laboratorio 5 — Primer Modelo de Clasificación
**Curso:** Minería de Datos (EIN132A25)

## Objetivos
- Preparar un dataset limpio para entrenar
- Entrenar un **árbol de decisión** (Decision Tree)
- Hacer predicciones y evaluar el modelo
- Visualizar el árbol entrenado

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
df.head()

## 1. Preparar el dataset

In [ ]:
# Limpieza
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df = df.drop(columns=["Cabin"])
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
embarked_dummies = pd.get_dummies(df["Embarked"], prefix="Embarked", drop_first=True)
df = pd.concat([df, embarked_dummies], axis=1)
df = df.drop(columns=["Embarked"])

features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked_Q", "Embarked_S"]
X = df[features]
y = df["Survived"]

print(f"Shape de X: {X.shape}")
print(f"Distribución del target:\n{y.value_counts()}")

## 2. Separar en Train y Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} muestras")
print(f"Test:  {X_test.shape[0]} muestras")

## 3. Entrenar el árbol de decisión

In [ ]:
modelo = DecisionTreeClassifier(max_depth=4, random_state=42)
modelo.fit(X_train, y_train)
print("Modelo entrenado.")
print(f"Profundidad del árbol: {modelo.get_depth()}")
print(f"Número de hojas: {modelo.get_n_leaves()}")

## 4. Predicciones y evaluación

In [ ]:
y_pred = modelo.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy en test: {accuracy:.4f} ({accuracy*100:.1f}%)") 

In [ ]:
print(classification_report(y_test, y_pred, target_names=["No sobrevivió", "Sobrevivió"]))

In [ ]:
acc_train = accuracy_score(y_train, modelo.predict(X_train))
acc_test  = accuracy_score(y_test, y_pred)
print(f"Accuracy TRAIN: {acc_train:.4f}")
print(f"Accuracy TEST:  {acc_test:.4f}")

## 5. Visualizar el árbol

In [ ]:
plt.figure(figsize=(20, 8))
plot_tree(modelo, feature_names=features, class_names=["No sobrevivió", "Sobrevivió"],
          filled=True, rounded=True, fontsize=10)
plt.title("Árbol de Decisión — Titanic")
plt.show()

## 6. Importancia de features

In [ ]:
importancias = pd.Series(modelo.feature_importances_, index=features).sort_values(ascending=False)
importancias.plot(kind="bar", color="steelblue")
plt.title("Importancia de cada feature")
plt.ylabel("Importancia")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Ejercicios

### Ejercicio 1 — Comparar max_depth=1 vs max_depth=10

In [ ]:
for depth in [1, 10]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42)
    m.fit(X_train, y_train)
    tr = accuracy_score(y_train, m.predict(X_train))
    te = accuracy_score(y_test, m.predict(X_test))
    print(f"max_depth={depth}: Train={tr:.4f}, Test={te:.4f}")

### Ejercicio 2 — Agregar FamilySize

In [ ]:
X2 = X.copy()
X2["FamilySize"] = X2["SibSp"] + X2["Parch"] + 1
features2 = features + ["FamilySize"]
X2_train, X2_test, _, _ = train_test_split(X2, y, test_size=0.2, random_state=42, stratify=y)
m2 = DecisionTreeClassifier(max_depth=4, random_state=42)
m2.fit(X2_train, y_train)
print(f"Accuracy con FamilySize: {accuracy_score(y_test, m2.predict(X2_test)):.4f}")

### Desafío — Curva de overfitting

In [ ]:
depths = range(1, 16)
train_scores = []
test_scores  = []

for depth in depths:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42)
    m.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, m.predict(X_train)))
    test_scores.append(accuracy_score(y_test, m.predict(X_test)))

plt.plot(depths, train_scores, label="Train", marker="o")
plt.plot(depths, test_scores, label="Test", marker="o")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Overfitting según profundidad del árbol")
plt.legend()
plt.grid(True)
plt.show()